# Projet Machine Learning

### Groupe 3

**Membres :**
- PONCHON Léo
- MOUFOK Sabrina
- HUANG Lei
- ZHANG Yaqi
- AYDEMIR Yagmur

## 0. Préparation

In [ ]:
# Librairies
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import chi2, SelectKBest

from sklearn.base import BaseEstimator, TransformerMixin # Estimator personels

import plotly.graph_objs as go #graphiques avancés plotly
import plotly.offline as py #mode offline plotly
import plotly.express as px #visualisation rapide
import plotly.io as pio #sauvegarde d'image

# Librairies NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer #lemmatisation
from nltk import sent_tokenize #decoupage en phrases
from nltk import word_tokenize #decoupage en mots
from nltk import pos_tag #etiquetage grammatical
from nltk.stem.snowball import SnowballStemmer # Stemmatisation
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords #stopwords

# Téléchargement des ressources NLTK
nltk.download('stopwords', quiet=True) #stopwords
nltk.download('wordnet', quiet=True) #WordNet
nltk.download('omw-1.4', quiet=True)


nltk.download("punkt") #tokenisation
nltk.download("punkt_tab") #tokenisation
nltk.download("averaged_perceptron_tagger") # Tags
nltk.download("averaged_perceptron_tagger_eng") # Tags anglais
nltk.download("tagsets_json") #Liste de tags
nltk.download("tagsets") #Ancienne liste

#import contractions
from collections import Counter # Comptage d'occurrences
from wordcloud import WordCloud # Generation de nuages de mots

from sklearn.decomposition import PCA # Réduction de dimension (ACP)
from mpl_toolkits.mplot3d import Axes3D # nécessaire pour la 3D
import plotly.express as px
from sklearn.manifold import TSNE

#Librairies UMAP
import umap

In [ ]:
# Chargement du dataset
df = pd.read_csv("./scitweets_export.tsv", sep="\t")

## 1. Comprendre les données

### 1.1 Aperçu global

In [ ]:
# Vérifier les colonnes
# Comprendre les colonnes
print(df.columns)
print(df.head())

In [ ]:
# Vérifier les données
print(df["science_related"].value_counts())

print(df[["scientific_claim",
    "scientific_reference",
    "scientific_context"]].astype(int).sum())

In [ ]:
# Incohérences
incoherence1 = df[(df["science_related"] == 0) & ((df["scientific_claim"] == 1) | (df["scientific_reference"] == 1) | (df["scientific_context"] == 1))]
print(f"Incohérence 1: {len(incoherence1)}")

incoherence2 = df[(df["science_related"] == 1) & ((df["scientific_claim"] == 0) & (df["scientific_reference"] == 0) & (df["scientific_context"] == 0))]
print(f"Incohérence 2: {len(incoherence2)}")

In [ ]:
# data frame de science related
df_sci = df[df["science_related"] == 1].copy()

# situation single-label
# {CLAIM}
claim = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 0) & (df_sci["scientific_context"] == 0)]

# {REF}
ref = df_sci[(df_sci["scientific_claim"] == 0) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 0)]

# {CONTEXT}
context = df_sci[(df_sci["scientific_claim"] == 0) & (df_sci["scientific_reference"] == 0) & (df_sci["scientific_context"] == 1)]

# situation multi-label
# {CLAIM + REF}
claim_ref = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 0)]

# {CLAIM + CONTEXT}
claim_context = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 0) & (df_sci["scientific_context"] == 1)]

# {REF + CONTEXT}
ref_context = df_sci[(df_sci["scientific_claim"] == 0) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 1)]

# {CLAIM + REF + CONTEXT}
all_three = df_sci[(df_sci["scientific_claim"] == 1) & (df_sci["scientific_reference"] == 1) & (df_sci["scientific_context"] == 1)]


print(f"Only Claim: {len(claim)}")
print(f"Only Reference: {len(ref)}")
print(f"Only Context: {len(context)}")
print(f"Claim + Ref: {len(claim_ref)}")
print(f"Claim + Context: {len(claim_context)}")
print(f"Ref + Context: {len(ref_context)}")
print(f"Claim + Ref + Context: {len(all_three)}")

print("-------------------------")
# task2 multi label
m_task2 = len(claim_context) + len(ref_context) + len(all_three)
# task multi label
m_task3 = len(claim_ref) + len(claim_context) + len(ref_context) + len(all_three)

print(f"Multi labels pour le task 2: {m_task2}")
print(f"Multi labels pour le task 3: {m_task3}")

### 1.2 Visualisation la distribution des labels

In [ ]:
# Vérifier l’équilibrage
df["science_related"].value_counts().plot(kind="bar")
plt.show()


In [ ]:
df["scientific_claim"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df["scientific_reference"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df["scientific_context"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df_sci["scientific_claim"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df_sci["scientific_reference"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
df_sci["scientific_context"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
label_statistique = {
    "Science Related": len(df_sci),
    "Only Claim": len(claim),
    "Only Ref": len(ref),
    "Only Context": len(context),
    "Claim + Ref": len(claim_ref),
    "Claim + Context": len(claim_context),
    "Ref + Context": len(ref_context),
    "All Three": len(all_three)
}

s = pd.Series(label_statistique)

plt.figure(figsize=(12, 6))
ax = s.plot(kind="bar")

plt.title("Distribution des label Combinatoires ", fontsize=14)
plt.xlabel("Combination de label", fontsize=12)
plt.ylabel("Nombre", fontsize=12)

for i, v in enumerate(s):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.show()

### 1.3 Les word clouds

In [ ]:
#Une première analyse des documents
#les Word Clouds
#sans nettoyage

# Garder seulement les textes non vides
texts = df["text"].dropna().astype(str)

# Concaténer tous les tweets en un seul gros texte
corpus = " ".join(texts)

# Découpage très simple
words = corpus.split()

# Comptage
word_counts = Counter(words)

# Word cloud
wc = WordCloud(width=1000, height=500, background_color="white")
wc.generate_from_frequencies(word_counts)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

# wc.to_file("./outputs/1-comprendre_les_données/worldcloud.png")

### 1.4 les emojis

In [ ]:
# Définition du pattern pour détecter les emojis dans le texte (aide de l'IA uniquement pour trouver les codes UTF des emojis)
emoji_pattern = re.compile(
    "[\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "]+",
    re.UNICODE
)

def extract_emojis_from_df(df):
    emojis = []

    # On parcourt chaque colonne de df
    for colonne in df.columns:
        # Puis chaque cellule de la colonne
        for valeur in df[colonne]:
            texte = str(valeur)

            # On cherche les emojis dans le texte
            emojis_trouves = emoji_pattern.findall(texte)

            # On les ajoute à la liste globale
            emojis.extend(emojis_trouves)

    return emojis


# Extraction
emojis = extract_emojis_from_df(df)

# Suppression des doublons en mettant dans un ensemble, puis tri
emojis_uniques = sorted(set(emojis))

# Affichage
print("\nEmojis uniques :", emojis_uniques)

## 2. Prétraitements

### 2.1 Fonction de nottoyage

In [ ]:
# Initialisation du lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialisation du stemmer
stemmer = PorterStemmer()

# stop words
stop_words = set(stopwords.words("english"))
custom_stopwords = {"rt", "amp", "http", "via", "im", "dont", "cant", "u", "ur", "re", "ve", "ll", "s", "oh"}
final_stop_words = stop_words.union(custom_stopwords)

# Dictionnaire pour transformer emojis en mots
emoji_dict = {
    '🏀': ' basketball ', '💪': ' strong ', '✋': ' stop ', '💃': ' dance ',
    '💖': ' love ', '💀': ' dead ', '😉': ' wink ', '😷': ' sick ',
    '😧': ' surprised ', '😐': ' neutral ', '☀': ' sunny ', '😎': ' cool ',
    '❤': ' love ', '😊': ' happy ', '😬': ' nervous ', '💃🏻': ' dancing ',
    '😜': ' playful ', '🤦🏽': ' facepalm ', '♀': ' female ', '🤟': ' love ',
    '💸': ' money ', '👀': ' eyes ', '😈': ' devil ', '💔': ' broken ',
    '💙': ' love ', '🌊': ' wave ', '♻': ' recycle ', '👊': ' fist ',
    '👇': ' down ', '🌍': ' earth ', '💕': ' love ', '😄': ' happy ',
    '♥': ' love ', '👧': ' girl ', '💗': ' love ', '🐦': ' bird ',
    '🤣': ' laughing ', '🙄': ' eye roll ', '☑': ' checked ', '☞': ' point ',
    '❌': ' wrong ', '🎥': ' movie ', '📲': ' phone ', '👏': ' clap ',
    '🌞': ' sun ', '😃': ' happy ', '🙆🏻': ' okay ', '♂': ' male ',
    '🙋': ' wave ', '🦁': ' lion ', '💛': ' love ', '🍹': ' drink ',
    '🍷': ' wine ', '🤬': ' angry ', '📸': ' camera ', '☹': ' sad ',
    '👌🏿': ' okay ', '👌🏻': ' okay ', '👌🏽': ' okay ', '👌🏼': ' okay ',
    '👌🏾': ' okay ', '🧡': ' love ', '🗽': ' statue ', '🌼': ' flower ',
    '🌱': ' plant ', '🌸': ' flower ', '🌺': ' flower ', '👁': ' eye ',
    '✨': ' sparkle ', '😍': ' love ', '💥': ' boom ', '✌🏼': ' victory ',
    '🙏🏼': ' pray ', '✈': ' plane ', '🙂': ' smile ', '🤢': ' sick ',
    '😅': ' nervous ', '✧': ' star ', '🍃': ' leaf ', '💐': ' bouquet ',
    '🍁': ' leaf ', '👉': ' point ', '➡': ' right ', '➔': ' right ',
    '💯': ' hundred ', '✔': ' check ', '➙': ' arrow ', '😢': ' sad ',
    '😡': ' angry '
}


def CleanText(X,
                remove_digit=False,
                remove_stopwords=False,
                getlemmatisation=False,
                getstemmer=False,
                ):

    # conversion en texte
    sentence = str(X)

    # remplacer les emojis par texte
    for e, word in emoji_dict.items():
        sentence = sentence.replace(e, word)

    # enlèver les URLs
    sentence = re.sub(r"http\S+|www\S+", " ", sentence)

    # enlèver les mentions
    sentence = re.sub(r"@\w+", " ", sentence)

    # suppression ponctuation / caractères spéciaux
    sentence = re.sub(r'[^\w\s]', ' ', sentence)

    # enlèver les espaces
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    # découpage en tokens
    tokens = word_tokenize(sentence)

    # minuscules
    tokens = [t.lower() for t in tokens]

    # suppression tokens non alphanumériques
    tokens = [t for t in tokens if t.isalnum()]

    # suppression chiffres
    if remove_digit:
        tokens = [t for t in tokens if not t.isdigit()]
    
    # suppression stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in final_stop_words]

    # lemmatisation
    if getlemmatisation:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    # stemming
    if getstemmer:
        tokens = [stemmer.stem(t) for t in tokens]

    return " ".join(tokens)

### 2.2 TextNormalizer

In [ ]:
class TextNormalizer(BaseEstimator, TransformerMixin):
    def __init__(self,
                 remove_digit=False,
                 remove_stopwords=False,
                 getlemmatisation=False,
                 getstemmer=False,
                 ):
        
        # intial parameters
        self.remove_digit = remove_digit
        self.remove_stopwords = remove_stopwords
        self.getlemmatisation = getlemmatisation
        self.getstemmer = getstemmer
    
    def fit(self, X, y=None):
        # Aucun apprentissage pour ce type de Transformer
        return self
    
    def transform(self, X):
        # Applique CleanText à chaque élément de X
        X = X.copy()
        return [
            CleanText(
                    text,
                    remove_digit = self.remove_digit,
                    remove_stopwords=self.remove_stopwords,
                    getlemmatisation=self.getlemmatisation,
                    getstemmer=self.getstemmer,
            )
            for text in X
        ]   

### 2.2 les Versions différentes de nettoyge

In [ ]:
pretaitement_config = {
    "V1_remove_digit": {
        "remove_digit": True,
    },
    "V2_stopwords": {
        "remove_digit": True,
        "remove_stopwords": True,
    },
    "V3_lemma": {
        "remove_digit": True,
        "remove_stopwords": True,
        "getlemmatisation": True,
    },
    "V4_stem": {
        "remove_digit": True,
        "remove_stopwords": True,
        "getstemmer": True,
    },
}

### 2.3 Comparaison les résultats de nettoyage

In [ ]:
text_cleaned = df[["text"]].copy()

for version_name, config in pretaitement_config.items():
    normalizer = TextNormalizer(**config)
    text_cleaned[f"text_clean_{version_name}"] = normalizer.transform(df["text"])

text_cleaned[["text", "text_clean_V1_remove_digit", "text_clean_V2_stopwords", "text_clean_V3_lemma", "text_clean_V4_stem"]].head(5)


In [ ]:
versions = list(pretaitement_config.keys())

# word cloud pour comparaison
n_cols = 2
n_rows = (len(versions) + n_cols - 1) // n_cols
plt.figure(figsize=(20, 6 * n_rows))

for i, v_name in enumerate(versions):
    col = f"text_clean_{v_name}"
    plt.subplot(n_rows,n_cols,i+1)

    corpus = " ".join(text_cleaned[col].dropna().astype(str))
    wc = WordCloud(width=800, height=400, background_color="white", stopwords=[]).generate(corpus)

    print("\n=======================")
    print(col + ":")

    raw_counts = wc.process_text(corpus)
    sorted_counts = sorted(raw_counts.items(), key=lambda item: item[1], reverse=True)
    for word, count in sorted_counts[:10]:
        print(f"{word}: {count}")       

    plt.imshow(wc)
    plt.title(col)
    plt.axis("off")

plt.show()

## 3. Définir les 3 tâches

### 3.1 Task 1: SCI vs NON0SCI

In [ ]:
def create_task1(df):
    task = df[df["science_related"].isin([0, 1])].copy()
    X = task["text"]
    y = task["science_related"].astype(int)
    return X, y

### 3.2 Task 2: CLAIM,REF vs CONTEXT

In [ ]:
# prorité pour label CLAIM/REF
def create_task2(df):
    task = df[df["science_related"] == 1].copy()
    X = task["text"]
    y = ((task["scientific_claim"] == 1) | (task["scientific_reference"] == 1)).astype(int)
    return X, y

def create_task2_multilabel(df):
    task = df[df["science_related"] == 1].copy()
    X = task["text"]
    task["claim_ref"] = ((task["scientific_claim"] == 1) | (task["scientific_reference"] == 1)).astype(int)
    y = task[["claim_ref", "scientific_context"]].values
    return X, y

### 3.3 Task 3: CLAIM vs REF vs CONTEXT

In [ ]:
def get_label_v1(row):
    if row["scientific_claim"] == 1:
        return 0
    elif row["scientific_reference"] == 1:
        return 1
    elif row["scientific_context"] == 1:
        return 2
    else:
        return -1
    
def get_label_v2(row):
    if row["scientific_reference"] == 1:
        return 1
    elif row["scientific_context"] == 1:
        return 2
    elif row["scientific_claim"] == 1:
        return 0
    else:
        return -1

# CLAIM > REF > CONTEXT
def create_task3_v1(df):
    task = df[df["science_related"] == 1].copy()
    X = task["text"]
    y = task.apply(get_label_v1, axis=1)
    return X, y

# REF > CONTEXT > CLAIM
def create_task3_v2(df):
    task = df[df["science_related"] == 1].copy()
    X = task["text"]
    y = task.apply(get_label_v2, axis=1)
    return X, y

def create_task3_multilabel(df):
    task = df[df["science_related"] == 1].copy()
    X = task["text"]
    y = task[["scientific_claim", "scientific_reference", "scientific_context"]].values
    return X, y

### 3.4 Task configutation

In [ ]:
all_tasks_config = {
    "Task1": create_task1,
    "Task2": create_task2,
    # "Task2_multilabel": create_task2_multilabel,
    # "Task3_v1": create_task3_v1,
    "Task3_v2": create_task3_v2,
    # "Task3_multilabel": create_task3_multilabel
}

### 3.5 la distribution des labels des tâches

In [ ]:
for name, fn in all_tasks_config.items():
    X, y = fn(df)
    print(f"\n{name}: {len(y)} samples")
    if len(y.shape) > 1 and y.shape[1] > 1:
        print("Multi-label distribution (column sums):")
        counts = y.sum(axis=0).astype(int)
        print(counts)
    else:
        print(y.value_counts().sort_index())

## 4. Vectorisation et Visualisation

### 4.1 les funtions de la visulation

In [ ]:
def plot_pca_2d_3d(X, y, task_name, prep_name):
    # Conversion si matrice creuse
    X_dense = X.toarray() if hasattr(X, "toarray") else X
    y = np.array(y)

    # ACP 2D
    pca_2d = PCA(n_components=2, random_state=42)
    X_pca_2d = pca_2d.fit_transform(X_dense)

    # ACP 3D
    pca_3d = PCA(n_components=3, random_state=42)
    X_pca_3d = pca_3d.fit_transform(X_dense)

    colors = ['#ff9999', '#66b3ff', '#99ff99']
    main_title = f"{task_name} | Pre-traitement: {prep_name}"

    # Figure
    fig = plt.figure(figsize=(12, 5))
    fig.suptitle(main_title, fontsize=14, fontweight='bold', y=1.05)
    
    # ----- Plot 2D -----
    ax1 = fig.add_subplot(1, 2, 1)
    for cls in np.unique(y):
        idx = (y == cls)
        ax1.scatter(X_pca_2d[idx, 0], X_pca_2d[idx, 1], 
                    alpha=0.6, color=colors[int(cls)], label=f"Classe {int(cls)}")

    ax1.set_title("ACP 2D")
    ax1.set_xlabel("PC1")
    ax1.set_ylabel("PC2")
    ax1.legend()

    # ----- Plot 3D -----
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    for cls in np.unique(y):
        idx = (y == cls)
        ax2.scatter(X_pca_3d[idx, 0], X_pca_3d[idx, 1], X_pca_3d[idx, 2],
                    alpha=0.6, color=colors[int(cls)], label=f"Classe {int(cls)}")

    ax2.set_title("ACP 3D")
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")
    ax2.set_zlabel("PC3")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_tsne_2d_3d(X, y, task_name, prep_name):
    # Conversion si matrice creuse
    X_dense = X.toarray() if hasattr(X, "toarray") else X
    y = np.array(y)
    
    # t-SNE 2D
    tsne_2d = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    X_tsne_2d = tsne_2d.fit_transform(X_dense)

    # t-SNE 3D
    tsne_3d = TSNE(n_components=3, random_state=42, init='pca', learning_rate='auto')
    X_tsne_3d = tsne_3d.fit_transform(X_dense)

    colors = ['#ff9999', '#66b3ff', '#99ff99']
    main_title = f"{task_name} | Pre-traitement: {prep_name}"

    # Figure
    fig = plt.figure(figsize=(12, 5))
    fig.suptitle(main_title, fontsize=14, fontweight='bold', y=1.05)
    
    # ----- Plot 2D -----
    ax1 = fig.add_subplot(1, 2, 1)
    for cls in np.unique(y):
        idx = (y == cls)
        ax1.scatter(X_tsne_2d[idx, 0], X_tsne_2d[idx, 1], 
                    alpha=0.6, color=colors[int(cls)], label=f"Classe {int(cls)}")

    ax1.set_title("t-SNE 2D")
    ax1.set_xlabel("Dimension 1") 
    ax1.set_ylabel("Dimension 2")
    ax1.legend()

    # ----- Plot 3D -----
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    for cls in np.unique(y):
        idx = (y == cls)
        ax2.scatter(X_tsne_3d[idx, 0], X_tsne_3d[idx, 1], X_tsne_3d[idx, 2],
                    alpha=0.6, color=colors[int(cls)], label=f"Classe {int(cls)}")

    ax2.set_title("t-SNE 3D")
    ax2.set_xlabel("Dimension 1")
    ax2.set_ylabel("Dimension 2")
    ax2.set_zlabel("Dimension 3")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_umap_2d_3d(X, y, task_name, prep_name):
    # Conversion si matrice creuse
    X_dense = X.toarray() if hasattr(X, "toarray") else X
    y = np.array(y)

    # UMAP 2D
    umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap_2d = umap_2d.fit_transform(X_dense)

    # UMAP 3D
    umap_3d = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap_3d = umap_3d.fit_transform(X_dense)

    colors = ['#ff9999', '#66b3ff', '#99ff99']
    main_title = f"{task_name} | Pre-traitement: {prep_name}"

    # Figure
    fig = plt.figure(figsize=(12, 5))
    fig.suptitle(main_title, fontsize=14, fontweight='bold', y=1.05)
    
    # ----- Plot 2D -----
    ax1 = fig.add_subplot(1, 2, 1)
    for cls in np.unique(y):
        idx = (y == cls)
        ax1.scatter(X_umap_2d[idx, 0], X_umap_2d[idx, 1], 
                    alpha=0.6, color=colors[int(cls)], label=f"Classe {int(cls)}")

    ax1.set_title("UMAP 2D")
    ax1.set_xlabel("UMAP 1") 
    ax1.set_ylabel("UMAP 2")
    ax1.legend()

    # ----- Plot 3D -----
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    for cls in np.unique(y):
        idx = (y == cls)
        ax2.scatter(X_umap_3d[idx, 0], X_umap_3d[idx, 1], X_umap_3d[idx, 2],
                    alpha=0.6, color=colors[int(cls)], label=f"Classe {int(cls)}")

    ax2.set_title("UMAP 3D")
    ax2.set_xlabel("UMAP 1")
    ax2.set_ylabel("UMAP 2")
    ax2.set_zlabel("UMAP 3")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

### 4.2 Task1 - vectorisation et visualisation

In [ ]:
X_raw_task1, y_task1 = create_task1(df)
for prep_name in pretaitement_config.keys():
        cleaner = TextNormalizer(**pretaitement_config[prep_name])
        vectorizer= TfidfVectorizer()
    
        X_clean = cleaner.fit_transform(X_raw_task1)
        X_vec = vectorizer.fit_transform(X_clean)

        plot_pca_2d_3d(X_vec, y_task1, "Task1", prep_name)
        # plot_tsne_2d_3d(X_vec, y, "Task1", prep_name)
        # plot_umap_2d_3d(X_vec, y, "Task1", prep_name)


### 4.3 Task2 - vectorisation et visualisation

In [ ]:
X_raw_task2, y_task2 = create_task2(df)
for prep_name in pretaitement_config.keys():
        cleaner = TextNormalizer(**pretaitement_config[prep_name])
        vectorizer= TfidfVectorizer()
    
        X_clean = cleaner.fit_transform(X_raw_task2)
        X_vec = vectorizer.fit_transform(X_clean)

        # plot_pca_2d_3d(X_vec, y_task2, "Task2", prep_name)
        plot_tsne_2d_3d(X_vec, y_task2, "Task2", prep_name)
        # plot_umap_2d_3d(X_vec, y_task2, "Task2", prep_name)

### 4.4 Task 3 - vectorisation et visualisation

In [ ]:
X_raw_task3, y_task3 = create_task3_v2(df)
for prep_name in pretaitement_config.keys():
        cleaner = TextNormalizer(**pretaitement_config[prep_name])
        vectorizer= TfidfVectorizer()
    
        X_clean = cleaner.fit_transform(X_raw_task3)
        X_vec = vectorizer.fit_transform(X_clean)

        # plot_pca_2d_3d(X_vec, y_task3, "Task3", prep_name)
        # plot_tsne_2d_3d(X_vec, y_task3, "Task3", prep_name)
        plot_umap_2d_3d(X_vec, y_task3, "Task3", prep_name)

## 5. Modélisation 

### 5.1 preparation

In [ ]:
classifiers = {
    "Naive Bayes":          MultinomialNB(),
    "SVM":                  LinearSVC(max_iter=5000, random_state=42),
    "Logistic Regression":  LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":        DecisionTreeClassifier(random_state=42),
    "KNN":                  KNeighborsClassifier(n_neighbors=5),
    "Random Forest":        RandomForestClassifier(n_estimators=100, random_state=42)
}

In [ ]:
def build_pipeline(preprocess_config, classifier):
    pipe =  Pipeline([
        ("cleaner", TextNormalizer(**preprocess_config)),
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2, max_df=0.95)),
        ("smote", SMOTE(random_state=42)),
        ("clf", classifier)
    ])
    return pipe

### 5.2 Task1 - choisir le meiller pre-traitement

In [ ]:
task1_pretaitement_results = []
X, y = create_task1(df)
    
for prep_name, prep_config in pretaitement_config.items():
    for clf_name, clf in classifiers.items():
            
        pipe = build_pipeline(prep_config, clf)
        scores = cross_validate(
            pipe, X, y, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
            scoring=['accuracy', 'f1_macro'], 
            error_score='raise'
        )

        task1_pretaitement_results.append({
            "Task": "Task1",
            "Pretraitement": prep_name,
            "Classifier": clf_name,
            "Accuracy_Mean": np.mean(scores['test_accuracy']),
            "Accuracy_Std": np.std(scores['test_accuracy']),
            "F1_Macro_Mean": np.mean(scores['test_f1_macro']),
            "F1_Macro_Std": np.std(scores['test_f1_macro']),
            "Fit_Time(s)": np.mean(scores['fit_time'])
        })

df_results = pd.DataFrame(task1_pretaitement_results)
df_results = df_results.sort_values(by=["Task", "F1_Macro_Mean"], ascending=[True, False])
print(df_results)
        

### 5.2 Task2 - choisir le meiller pre-traitement

In [ ]:
task2_pretaitement_results = []
X, y = create_task2(df)
    
for prep_name, prep_config in pretaitement_config.items():
    for clf_name, clf in classifiers.items():
            
        pipe = build_pipeline(prep_config, clf)
        scores = cross_validate(
            pipe, X, y, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
            scoring=['accuracy', 'f1_macro'], 
            error_score='raise'
        )

        task2_pretaitement_results.append({
            "Task": "Task2",
            "Pretraitement": prep_name,
            "Classifier": clf_name,
            "Accuracy_Mean": np.mean(scores['test_accuracy']),
            "Accuracy_Std": np.std(scores['test_accuracy']),
            "F1_Macro_Mean": np.mean(scores['test_f1_macro']),
            "F1_Macro_Std": np.std(scores['test_f1_macro']),
            "Fit_Time(s)": np.mean(scores['fit_time'])
        })

df_results = pd.DataFrame(task2_pretaitement_results)
df_results = df_results.sort_values(by=["Task", "F1_Macro_Mean"], ascending=[True, False])
print(df_results)

In [ ]:
task2_pretaitement_results = []
X, y = create_task2_multilabel(df)
    
for prep_name, prep_config in pretaitement_config.items():
    for clf_name, clf in classifiers.items():
            
        pipe = build_pipeline(prep_config, clf)
        scores = cross_validate(
            pipe, X, y, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
            scoring=['accuracy', 'f1_macro'], 
            error_score='raise'
        )

        task2_pretaitement_results.append({
            "Task": "Task2",
            "Pretraitement": prep_name,
            "Classifier": clf_name,
            "Accuracy_Mean": np.mean(scores['test_accuracy']),
            "Accuracy_Std": np.std(scores['test_accuracy']),
            "F1_Macro_Mean": np.mean(scores['test_f1_macro']),
            "F1_Macro_Std": np.std(scores['test_f1_macro']),
            "Fit_Time(s)": np.mean(scores['fit_time'])
        })

df_results = pd.DataFrame(task2_pretaitement_results)
df_results = df_results.sort_values(by=["Task", "F1_Macro_Mean"], ascending=[True, False])
print(df_results)

### 5.3 Task3 - choisir le meiller pre-traitement

In [ ]:
task3_pretaitement_results = []
X, y = create_task3_v2(df)
    
for prep_name, prep_config in pretaitement_config.items():
    for clf_name, clf in classifiers.items():
            
        pipe = build_pipeline(prep_config, clf)
        scores = cross_validate(
            pipe, X, y, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
            scoring=['accuracy', 'f1_macro'], 
            error_score='raise'
        )

        task3_pretaitement_results.append({
            "Task": "Task3",
            "Pretraitement": prep_name,
            "Classifier": clf_name,
            "Accuracy_Mean": np.mean(scores['test_accuracy']),
            "Accuracy_Std": np.std(scores['test_accuracy']),
            "F1_Macro_Mean": np.mean(scores['test_f1_macro']),
            "F1_Macro_Std": np.std(scores['test_f1_macro']),
            "Fit_Time(s)": np.mean(scores['fit_time'])
        })

df_results = pd.DataFrame(task3_pretaitement_results)
df_results = df_results.sort_values(by=["Task", "F1_Macro_Mean"], ascending=[True, False])
print(df_results)

In [ ]:
task3_v1_pretaitement_results = []
X, y = create_task3_v1(df)
    
for prep_name, prep_config in pretaitement_config.items():
    for clf_name, clf in classifiers.items():
            
        pipe = build_pipeline(prep_config, clf)
        scores = cross_validate(
            pipe, X, y, 
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
            scoring=['accuracy', 'f1_macro'], 
            error_score='raise'
        )

        task3_v1_pretaitement_results.append({
            "Task": "Task3_v1",
            "Pretraitement": prep_name,
            "Classifier": clf_name,
            "Accuracy_Mean": np.mean(scores['test_accuracy']),
            "Accuracy_Std": np.std(scores['test_accuracy']),
            "F1_Macro_Mean": np.mean(scores['test_f1_macro']),
            "F1_Macro_Std": np.std(scores['test_f1_macro']),
            "Fit_Time(s)": np.mean(scores['fit_time'])
        })

df_results = pd.DataFrame(task3_v1_pretaitement_results)
df_results = df_results.sort_values(by=["Task", "F1_Macro_Mean"], ascending=[True, False])
print(df_results)